# Assignment 3: Milestone I – Natural Language Processing
## Task 1: Basic Text Pre-processing
#### Student Name: XXXX XXXX
#### Student ID: 000000

**Environment:** Python 3 and Jupyter Notebook

**Libraries used:**
- `pandas` – data loading and manipulation
- `re` – regular expression tokenization
- `collections.Counter` – computing term and document frequencies

## Introduction

This notebook implements a complete text pre-processing pipeline for a dataset of cosmetics and beauty product reviews (~61,000 reviews). The goal is to clean and normalise the `review_text` field so it can be used for downstream NLP tasks such as feature representation and classification.

The pipeline follows these steps in order:
1. Load and examine the dataset
2. Tokenize each review using a specified regular expression
3. Convert all tokens to lowercase
4. Remove tokens shorter than 2 characters
5. Remove stopwords using the provided `stopwords_en.txt`
6. Remove words that appear only once across all reviews (term frequency = 1)
7. Remove the top 20 most frequent words by document frequency
8. Save the cleaned data to `processed.csv` and build the vocabulary in `vocab.txt`

## Importing Libraries

In [1]:
import pandas as pd          # for loading and manipulating the dataset
import re                    # for regex-based tokenization
from collections import Counter  # for computing term and document frequencies

## 1.1 Examining and Loading Data

We begin by loading a small sample of the dataset to understand its structure, then load the full dataset for processing.

**Dataset:** ~61,000 cosmetics and beauty product reviews from Nykaa.

**Key columns used in this task:**
- `review_text` – the full text body of the review (pre-processed in Task 1)
- `review_title` – the title of the review (used later in Task 3)
- `is_a_buyer` – boolean label: `True` = buyer, `False` = non-buyer (target for classification)

In [2]:
# Load a small sample to inspect structure before loading the full dataset
sample_df = pd.read_csv('cosmetics_beauty_products_reviews.csv', nrows=5)

print('Column names:', sample_df.columns.tolist())
print('\nData types:')
print(sample_df.dtypes)
print('\nSample rows (key columns):')
sample_df[['review_title', 'review_text', 'is_a_buyer']]

Column names: ['product_id', 'brand_name', 'review_id', 'review_title', 'review_text', 'author', 'review_date', 'review_rating', 'is_a_buyer', 'product_title', 'price', 'avg_product_rating', 'product_rating_count', 'product_tags', 'product_url']

Data types:
product_id                int64
brand_name               object
review_id                 int64
review_title             object
review_text              object
author                   object
review_date              object
review_rating             int64
is_a_buyer                 bool
product_title            object
price                     int64
avg_product_rating      float64
product_rating_count      int64
product_tags            float64
product_url              object
dtype: object

Sample rows (key columns):


,review_title,review_text,is_a_buyer
0,Worth buying 50g one,Works as it claims. Could see the difference f...,True
1,Best cream to start ur day,It does what it claims . Best thing is it smoo...,True
2,perfect for summers dry for winters,I have been using this product for months now....,True
3,Not a moisturizer,"i have an oily skin, while this whip acts as a...",True
4,Average,It's not that good. Please refresh try for oth...,True


In [3]:
# Load the full dataset
df = pd.read_csv('cosmetics_beauty_products_reviews.csv')

print(f'Total number of reviews: {len(df)}')
print(f'Number of columns: {len(df.columns)}')
print(f'\nMissing values in review_text: {df["review_text"].isnull().sum()}')
print(f'Missing values in review_title: {df["review_title"].isnull().sum()}')
print(f'\nClass distribution (is_a_buyer):')
print(df['is_a_buyer'].value_counts())
print(f'\nClass balance ratio: {df["is_a_buyer"].value_counts(normalize=True).round(3).to_dict()}')

Total number of reviews: 61284
Number of columns: 15

Missing values in review_text: 9
Missing values in review_title: 0

Class distribution (is_a_buyer):
is_a_buyer
True     48222
False    13062
Name: count, dtype: int64

Class balance ratio: {True: 0.787, False: 0.213}


**Findings:**
- The dataset contains 61,284 reviews across 15 columns.
- There are 9 missing values in `review_text`, which we will handle by treating them as empty strings.
- The dataset is imbalanced: ~78.7% of reviews are from buyers (`True`) and ~21.3% from non-buyers (`False`). This is important context for the classification tasks in Task 3.
- We focus only on `review_text` for Task 1 pre-processing, as specified.

## 1.2 Pre-processing Data

We apply the following pre-processing pipeline **only to `review_text`**. Each step is performed in sequence, as later steps depend on earlier ones.

### Step 1: Load Stopwords

We load the provided stopword list from `stopwords_en.txt`. As discussed in the course activities, there is no universal stopword list — the provided one is specific to this assignment.

In [4]:
# Load stopwords from the provided file into a set for O(1) lookup
with open('stopwords_en.txt', 'r') as f:
    stopwords_en = set(line.strip() for line in f if line.strip())

print(f'Total stopwords loaded: {len(stopwords_en)}')
print(f'Sample stopwords: {sorted(list(stopwords_en))[:15]}')

Total stopwords loaded: 570
Sample stopwords: ['a', "a's", 'able', 'about', 'above', 'according', 'accordingly', 'across', 'actually', 'after', 'afterwards', 'again', 'against', "ain't", 'all']


### Steps 2–5: Tokenize, Lowercase, Remove Short Words, Remove Stopwords

We define a single function that chains these four steps:

- **Tokenization**: Uses the required regex `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"` — this captures standard words as well as hyphenated words (e.g. `well-known`) and words with apostrophes (e.g. `it's`). As covered in Activity 1, this is more precise than a whitespace tokenizer.
- **Lowercase**: Normalises case so `Good` and `good` are treated identically (as covered in Activity 2).
- **Short word removal**: Words of length < 2 are noise (e.g. single letters left after tokenization).
- **Stopword removal**: Common function words (e.g. `the`, `is`, `and`) are removed as they carry little semantic information for classification.

In [5]:
# Required tokenization regex as specified in the assignment
TOKEN_PATTERN = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"

def tokenize_and_clean(text):
    """
    Apply steps 2-5 of pre-processing to a single review text:
      - Tokenize using the specified regex
      - Convert to lowercase
      - Remove tokens with length < 2
      - Remove stopwords
    Returns a list of cleaned tokens.
    """
    if not isinstance(text, str):
        return []  # handle missing/non-string values
    
    # Step 2: Tokenize using the required regular expression
    tokens = re.findall(TOKEN_PATTERN, text)
    
    # Step 3: Convert all tokens to lowercase
    tokens = [t.lower() for t in tokens]
    
    # Step 4: Remove tokens with length less than 2
    tokens = [t for t in tokens if len(t) >= 2]
    
    # Step 5: Remove stopwords using the provided stopword list
    tokens = [t for t in tokens if t not in stopwords_en]
    
    return tokens

# Handle missing values in review_text before applying tokenization
df['review_text'] = df['review_text'].fillna('')

# Apply tokenization pipeline to all reviews
df['tokens'] = df['review_text'].apply(tokenize_and_clean)

print('Tokenization complete.')
print(f'\nExample — original text:')
print(f'  "{df["review_text"].iloc[2]}"')
print(f'\nAfter tokenization and cleaning:')
print(f'  {df["tokens"].iloc[2]}')

Tokenization complete.

Example — original text:
  "I have been using this product for months now.. it is perfect for combination n oily skin as it is non greasy absorbs quickly and moisturises well but it doesnt work for winters"

After tokenization and cleaning:
  ['product', 'months', 'perfect', 'combination', 'oily', 'skin', 'greasy', 'absorbs', 'quickly', 'moisturises', 'doesnt', 'work', 'winters']


### Step 6: Remove Words Appearing Only Once (Term Frequency = 1)

**Term frequency (TF)** counts the total number of times each word appears across the entire collection. Words that appear only once provide no pattern for the model to learn from and can inflate the vocabulary with noise (typos, rare proper nouns, etc.).

In [6]:
# Compute term frequency: total count of each word across all documents
term_freq = Counter()
for tokens in df['tokens']:
    term_freq.update(tokens)  # counts all occurrences (not unique per doc)

print(f'Unique tokens before rare word removal: {len(term_freq)}')

# Identify words with term frequency of exactly 1
rare_words = {word for word, count in term_freq.items() if count == 1}
print(f'Words appearing only once (to remove): {len(rare_words)}')
print(f'Sample rare words: {sorted(list(rare_words))[:15]}')

Unique tokens before rare word removal: 15808
Words appearing only once (to remove): 7734
Sample rare words: ['a-stick', 'a-u', 'aaaahhhhhh', 'aaammmazzzing', 'aahh', 'aain', 'aal', 'aamezing', 'aand', 'aap', 'aapp', 'aargan', 'aat', 'aaway', 'aawweesooommeee']


In [7]:
# Remove rare words from each review's token list
df['tokens'] = df['tokens'].apply(lambda tokens: [t for t in tokens if t not in rare_words])

print('Rare words removed successfully.')
print(f'Remaining unique tokens: {len(set(t for tokens in df["tokens"] for t in tokens))}')

Rare words removed successfully.
Remaining unique tokens: 8074


### Step 7: Remove Top 20 Most Frequent Words by Document Frequency

**Document frequency (DF)** counts how many unique documents (reviews) contain each word. Words with very high document frequency appear in nearly every review and provide little discriminatory power for classification — they are essentially domain-level stopwords (e.g. `skin`, `product`, `good` in beauty reviews).

In [8]:
# Compute document frequency: number of reviews each word appears in
# Use set() so each word is counted only once per review
doc_freq = Counter()
for tokens in df['tokens']:
    doc_freq.update(set(tokens))  # set() ensures one count per document

# Display the top 20 most frequent words by document frequency
print('Top 20 most frequent words by document frequency:')
print(f'{"Word":<15} {"Doc Frequency"}')
print('-' * 30)
for word, count in doc_freq.most_common(20):
    print(f'{word:<15} {count}')

Top 20 most frequent words by document frequency:
Word            Doc Frequency
------------------------------
good            13645
product         11829
skin            8339
love            8301
shade           8001
nice            5774
hair            5601
amazing         4753
long            4619
perfect         4575
colour          4352
smooth          4042
color           4029
great           3698
beautiful       3598
loved           3590
easy            3516
buy             3246
time            3172
nykaa           3167


In [9]:
# Collect the top 20 words to remove
top20_words = {word for word, _ in doc_freq.most_common(20)}
print(f'Words to be removed: {top20_words}')

# Remove top 20 high-frequency words from each review's token list
df['tokens'] = df['tokens'].apply(lambda tokens: [t for t in tokens if t not in top20_words])

print('\nTop 20 high-frequency words removed successfully.')

Words to be removed: {'shade', 'hair', 'skin', 'color', 'buy', 'smooth', 'great', 'product', 'love', 'time', 'nice', 'good', 'nykaa', 'colour', 'long', 'easy', 'loved', 'amazing', 'perfect', 'beautiful'}

Top 20 high-frequency words removed successfully.


### Pre-processing Summary Statistics

In [10]:
# Final vocabulary after all filtering steps
final_term_freq = Counter()
for tokens in df['tokens']:
    final_term_freq.update(tokens)

token_lengths = df['tokens'].apply(len)

print('=' * 45)
print('     Pre-processing Summary')
print('=' * 45)
print(f'Total reviews processed      : {len(df)}')
print(f'Final vocabulary size        : {len(final_term_freq)}')
print(f'Reviews with no tokens left  : {token_lengths.eq(0).sum()}')
print(f'Avg tokens per review        : {token_lengths.mean():.1f}')
print(f'Max tokens in a review       : {token_lengths.max()}')
print(f'Min tokens in a review       : {token_lengths.min()}')
print('=' * 45)

     Pre-processing Summary
Total reviews processed      : 61284
Final vocabulary size        : 8054
Reviews with no tokens left  : 1911
Avg tokens per review        : 7.1
Max tokens in a review       : 133
Min tokens in a review       : 0


## Saving Required Outputs

### Save `processed.csv`

We save the full dataframe with the `review_text` column replaced by the joined cleaned tokens. All other columns are preserved for use in Task 3 (e.g. `review_title`, `price`, `avg_product_rating`, `is_a_buyer`).

In [11]:
# Replace review_text with space-joined cleaned tokens
processed_df = df.copy()
processed_df['review_text'] = processed_df['tokens'].apply(lambda tokens: ' '.join(tokens))
processed_df = processed_df.drop(columns=['tokens'])  # drop helper column

# Save to file
processed_df.to_csv('processed.csv', index=False)

print('processed.csv saved successfully.')
print(f'Shape: {processed_df.shape}')
print(f'\nSample processed review_text:')
print(processed_df['review_text'].iloc[2])

processed.csv saved successfully.
Shape: (61284, 15)

Sample processed review_text:
months combination oily greasy absorbs quickly moisturises doesnt work winters


### Save `vocab.txt`

The vocabulary is built from all words remaining after pre-processing. Words are sorted **alphabetically** and assigned an integer index starting from **0**. Format: `word:index` (one per line).

This vocabulary will be used in Task 2 to build count vector representations.

In [12]:
# Sort vocabulary alphabetically and assign index starting from 0
vocab_words = sorted(final_term_freq.keys())

print(f'Total vocabulary size: {len(vocab_words)}')
print(f'First 5 words : {vocab_words[:5]}')
print(f'Last 5 words  : {vocab_words[-5:]}')

Total vocabulary size: 8054
First 5 words : ['aa', 'aback', 'abd', 'abh', 'ability']
Last 5 words  : ['zipper', 'zips', 'zit', 'zits', 'zone']


In [13]:
# Save vocab.txt in required format: word:index (index starts from 0)
with open('vocab.txt', 'w') as f:
    for idx, word in enumerate(vocab_words):
        f.write(f'{word}:{idx}\n')

print('vocab.txt saved successfully.')
print('\nFirst 15 entries in vocab.txt:')
with open('vocab.txt', 'r') as f:
    for i, line in enumerate(f):
        if i >= 15:
            break
        print(f'  {line.strip()}')

vocab.txt saved successfully.

First 15 entries in vocab.txt:
  aa:0
  aback:1
  abd:2
  abh:3
  ability:4
  abit:5
  abnormally:6
  abroad:7
  abs:8
  absent:9
  absolute:10
  absolutely:11
  absorb:12
  absorbed:13
  absorbing:14


## Summary

In this task, we built a complete text pre-processing pipeline for the cosmetics and beauty product reviews dataset. Processing was applied exclusively to the `review_text` column as required.

**Key decisions and findings:**

- **Tokenization regex** `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"`: This pattern captures plain words as well as hyphenated and apostrophe-containing words (e.g. `well-known`, `it's`), which is more appropriate for review text than a simple whitespace or `\w+` tokenizer.

- **Stopword removal** using the provided `stopwords_en.txt` (570 words): Removed high-frequency function words that carry little semantic meaning for classification.

- **Rare word removal** (7,734 words with TF=1): These words (often typos or extremely niche terms) appear too infrequently to be useful features and would significantly inflate the vocabulary.

- **Top 20 document-frequency removal**: Words like `skin`, `product`, `good`, `love` appear in thousands of reviews and are effectively domain-level stopwords for beauty reviews — they do not help distinguish buyers from non-buyers.

- **Final vocabulary**: 8,054 unique words, sorted alphabetically and indexed from 0, saved in `vocab.txt`. The processed reviews are saved in `processed.csv` for use in Tasks 2 and 3.